# Core Data Workflow

We begin by using the WRDS query to gather info on the following SIC codes:
* 3674 — Semiconductors
* 3672 — Printed Circuit Boards
* 3679 — Electronic Components
* 2836 — Biological Products / Pharmaceuticals
* 2834 — Pharmaceutical Preparations
* 8731 — Commercial Physical & Biological Research
* 3571 — Electronic Computers
* 3572 — Computer Storage Devices
* 7372 — Prepackaged Software
* 7371 — Computer Programming Services

These are stored in our *raw_compustat_sic_pull* file. 

In [53]:
import pandas as pd

In [54]:

raw_compustat_pull = pd.read_csv("csv_data/raw_compustat_sic_pull.csv")
print(raw_compustat_pull.shape)

(13394, 21)


We now run a consolidation scrypt that isolates the companies with a market cap larger than $1 billion.

In [55]:
%run processes/market_cap_consolidation.py

In [56]:
compustat_large_cap = pd.read_csv("csv_data/compustat_large_cap.csv")
print(compustat_large_cap.shape)

(1463, 23)


In [57]:
compustat_large_cap.groupby('industry')['tic'].nunique().reset_index().rename(columns={'tic': 'unique_tickers'})

,industry,unique_tickers
0,Biotech,85
1,Semiconductors/Hardware,75
2,Software,59


In [58]:
compustat_large_cap.to_csv("csv_data/compustat_large_cap_v2.csv", index=False)

In [59]:
%run processes/ticker_txt_conversion.py

Now we are going to run a scrypt that takes our tickers and finds the BoardEx ids.

In [ ]:
boardex_id = pd.read_csv("csv_data/boardex_id_data.csv")
boardex_id.head(1)

,ticker,boardname,boardid
0,ATVI,ACTIVISION BLIZZARD INC (De-listed 10/2023),725


In [64]:
%run processes/boardid_txt_conversion.py

### 3. [WRDS] Insert tickers into BoardEx and extract key personnel from 2015-2025

[a] Go to WRDS > BoardEx > North America > Organization Summary > Composition of Officers, Directors, & Senior Managers

[b] Date range of 2015 to end of 2025. Apply codes by ticker and input txt file copied from our downloaded csv and pasted into textedit for txt file conversion.

[c] Choose query variables (all) about officers and run query for csv file. Download csv file into local folder for input in step **4**.

In [70]:
personnel = pd.read_csv("csv_data/total_board.csv")

In [71]:
len(personnel)

21210

In [84]:
ceos = personnel[personnel['rolename'].str.contains('ceo', case=False, na=False)]
ceos = ceos.sort_values('dateendrole', ascending=False) # want the most recent entries for each CEO first
ceos.head()

,companyid,datestartrole,directorid,directorname,companyname,rolename,dateendrole,datestartroleflag,dateendroleflag,seniority
12084,29816,2024-01-01,1864086,Sassine Ghazi,SYNOPSYS INC,President/CEO,9000-01-01,10,40,Executive Director
5180,16949,2024-06-05,1931869,Doctor Mark Gitin,IPG PHOTONICS CORP,CEO,9000-01-01,10,40,Executive Director
10850,26104,2023-12-11,340014,John Giamatteo,BLACKBERRY LTD (Research In Motion Ltd prior t...,CEO/Division President,9000-01-01,10,40,Executive Director
11017,27025,2020-08-17,83652,Jure Sola,SANMINA CORP (Sanmina-SCI Corp prior to 11/2012),Chairman/CEO,9000-01-01,10,40,Executive Director
11119,27081,2024-12-23,1518414,Carl Colizza,SAPUTO INC,President/CEO/Interim COO,9000-01-01,10,40,Senior Manager


In [85]:
ceos = ceos[ceos['seniority'] == 'Executive Director']

In [86]:
len(ceos)

468

In [87]:
ceos.to_csv("csv_data/ceos.csv", index=False)

In [88]:
%run processes/directorid_txt_conversion.py

In [89]:
raw_education = pd.read_csv('csv_data/boardex_raw_educ_pull.csv')
len(raw_education)

731

In [90]:
print(raw_education['qualification'].unique().tolist())

['Chartered Accountant', 'Degree', 'Certified Public Accountant', 'MBA (Distinction)', 'BSc (Hons)', 'BS', 'Fellow', 'MA', 'MBA', 'Diploma', 'BA', 'JD', 'Attended', 'MS', 'Doctorate (Hons)', 'PhD', 'BS (cum laude)', 'MA (Hons)', 'MD', 'BA (Hons)', 'Stanford Executive Program', 'BSEE', 'MSEE', 'JD (Hons)', 'BBA', 'JD (summa Cum Laude)', 'BS (Hons)', 'Graduated', 'Doctor of Humane Letters', 'Postdoctoral Fellow', 'AB', 'MSc', 'BSc', "Bachelor's Degree", 'Studied', 'BSME', 'MSME', 'AB (magna cum laude)', 'BSEE (magna cum laude)', 'BA (summa cum laude)', 'Certified', 'AB (Hons)', 'MEng', 'Post Graduate Diploma', 'BCom', 'Executive Program', 'Doctor of Science', 'Executive Development Program', 'Training Program', 'BSE', 'BSc (cum laude)', 'Certified Accountant', 'BS (Distinction)', 'Doctorate', 'Professional Development Program (PDP)', 'ME', 'BS (magna Cum Laude)', 'AA', 'Bachelor of Applied Science', 'BS (summa Cum Laude)', 'LLM', 'Completed', 'Postgraduate Studies', 'Certified Project Ma

In [92]:
ug_quals = {
    'Degree', 'BSc (Hons)', 'BS', 'BA', 'BS (cum laude)', 'BA (Hons)',
    'BSEE', 'BBA', 'BS (Hons)', 'AB', 'BSc', "Bachelor's Degree", 'BSME',
    'AB (magna cum laude)', 'BSEE (magna cum laude)', 'BA (summa cum laude)',
    'AB (Hons)', 'BCom', 'BSE', 'BSc (cum laude)', 'BS (Distinction)',
    'BS (magna Cum Laude)', 'BS (summa Cum Laude)', 'BTech',
    'Bachelor of Applied Science', 'BE (Hons)', 'Bachelor of Technology',
    "Bachelor's Degree (magna cum laude)", 'BEng (Hons)', 'LLB', 'BEng',
    'BPhil', 'BBA (Hons)', 'BBA (magna cum laude)', 'BE',
    "Bachelor's Degree (Hons)", 'BBA (summa cum laude)', 'AA'
}

mba_quals = {
    'MBA', 'MBA (Distinction)', 'MBA (Hons)', 'MBA (summa cum laude)',
    'International Executive MBA', 'BBA'
}

masters_quals = {
    'MA', 'MS', 'MA (Hons)', 'MSEE', 'MSc', 'MEng', 'MSME',
    'Masters Degree', 'Master of Management (MM)', 'LLM',
    'Master of Science (MOS)', 'MSc (magna cum laude)',
    'MSc (summa cum laude)', 'MBChB'
}

phd_quals = {
    'PhD', 'Doctorate', 'Doctorate (Hons)', 'Doctor of Humane Letters',
    'Doctor of Science', 'DSc', 'Doctor of Law (Hons)',
    'PhD (summa cum laude)', 'Doctor of Medicine (DM)'
}

md_quals = {
    'MD', 'MD (Hons)', 'Doctor of Veterinary Medicine (DVM)',
    'Bachelor of Medical Sciences (BMS)'
}

records = {}

for index, row in raw_education.iterrows():
    director_id = row['directorid']
    qual = row['qualification']
    company = row['companyname']
    value = f"{qual} {company}"

    if director_id not in records:
        records[director_id] = {'directorid': director_id}

    if qual in ug_quals:
        records[director_id]['UG'] = value
    elif qual in mba_quals:
        records[director_id]['MBA'] = value
    elif qual in phd_quals:
        records[director_id]['PhD'] = value
    elif qual in md_quals:
        records[director_id]['MD'] = value
    elif qual in masters_quals:
        records[director_id]["Master's"] = value

education = pd.DataFrame.from_dict(records, orient='index').reset_index(drop=True)

In [97]:
education.sort_values(by='directorid', inplace=True)
len(education)

318